# Real-Time Mode — a one-line trigger change

Spark 4.1 ships **Real-Time Mode**: sub-second, continuous structured streaming.
Turning it on is a **single line** — swap the trigger. Same query, same guardrails.

Here a stream of order events flows through stateless guardrails
(`HIGH_TOTAL`, `TOO_MANY_ITEMS`, `LEAKED_SECRET`) and is routed `ALLOW` / `QUARANTINE`.

In [ ]:
# Setup: Spark (with the RTM sink-allowlist bypass so we can use a live console
# sink), a Kafka topic, and a background producer dripping order events.
import sys, time
sys.path.insert(0, '/home/jovyan/demos/realtime-mode')
import rtm_stream_lib as rtm
from pyspark.sql import SparkSession
spark = (SparkSession.builder
    .config('spark.sql.streaming.realTimeMode.allowlistCheck','false')
    .getOrCreate())
rtm.start_producer(rows_per_sec=20)   # daemon thread -> kafka topic 'rtm-orders'
print('producer started')

### The one-line difference
```python
# micro-batch:   writer.trigger(processingTime='5 seconds').start()
# real-time mode: writer._jwrite.trigger(Trigger.RealTime('5 seconds')).start()
```
`start_query(spark, use_realtime=...)` flips exactly that one line.

In [ ]:
# 1) Baseline: ordinary micro-batch trigger.
q = rtm.start_query(spark, use_realtime=False)
time.sleep(16); q.stop()
print('micro-batch run complete')

In [ ]:
# 2) THE ONE-LINE CHANGE -> Real-Time Mode. Same query, same guardrails.
q = rtm.start_query(spark, use_realtime=True)
time.sleep(16); q.stop()
print('real-time mode run complete')